<div align="center">
<img src="https://poorit.in/image.png" alt="Poorit" width="40" style="vertical-align: middle;"> <b>LPU — BACKEND & GENERATIVE AI</b>

## Day 2 (b) — Company Pamphlet Generator

**Lovely Professional University**
*Backend & Generative AI · Poorit Technologies*

</div>

---

### What You'll Build

Type a company name and its website URL → your app **fetches the landing page**, hands that
text to the model, and **streams back a short pamphlet** for prospective customers,
investors and recruits.

```
name + URL  →  scrape the page  →  model writes the pamphlet  →  streams into the UI
```

1. Set up your API key — **you write this**
2. Run the scraper — it's given
3. Write the system message and the prompt
4. Stream the answer with **LiteLLM**
5. Wrap the whole thing in a Gradio UI
6. *(stretch)* let the user pick the tone

> **Needs an OpenAI API key.** Note what is *not* AI here: fetching the page is plain
> Python. The model only ever sees text you handed it.

---

## 1. Setup

Run these two cells. Nothing to write yet.

In [ ]:
# PROVIDED - just run this cell.
!pip install -q litellm gradio requests beautifulsoup4

In [ ]:
# PROVIDED - just run this cell.
import os
from getpass import getpass
import gradio as gr
import requests
from bs4 import BeautifulSoup
from litellm import completion

---

## 2. Your API key

`getpass` asks for the key at runtime, so it never gets written into the notebook file.

LiteLLM takes no key argument — it reads **`OPENAI_API_KEY` from the environment**, so
setting that is your job.

> Keys are personal. Never paste one into a code cell, never commit one.

In [ ]:
# Step 1 - Read your key with getpass and put it in os.environ["OPENAI_API_KEY"].
# Then set MODEL = "openai/gpt-4o-mini"  (LiteLLM wants the provider prefix).



---

## 3. The scraper — given

`requests` downloads the HTML, BeautifulSoup throws away the tags and hands back readable
text. It's cut to 2,000 characters so the prompt stays small and cheap.

Run both cells; there is nothing to write.

In [ ]:
# PROVIDED - just run this cell.

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}


def fetch_website_contents(url, max_chars=2_000):
    """Return the title and visible text of the page at `url`."""
    try:
        response = requests.get(url, headers=HEADERS, timeout=10)
        soup = BeautifulSoup(response.content, "html.parser")

        title = soup.title.string if soup.title else "No title found"

        if soup.body:
            for tag in soup.body(["script", "style", "img", "input"]):
                tag.decompose()                                  # drop the noise
            text = soup.body.get_text(separator="\n", strip=True)
        else:
            text = ""

        return (title + "\n\n" + text)[:max_chars]
    except Exception as e:
        return f"Error fetching website: {e}"


print("Scraper ready.")

In [ ]:
# PROVIDED - run it, then change the URL and run it again.

page = fetch_website_contents("https://anthropic.com")
print(page[:300])

---

## 4. Build the generator

Each cell below is one small step, and each one runs on its own — you don't need the UI to
exist before you can test the pieces.

> **Reminder:** a LiteLLM reply comes back as `response.choices[0].message.content`.
> When streaming, each piece is `chunk.choices[0].delta.content`.

In [ ]:
# Step 2 - Write pamphlet_system: ask for a short pamphlet, in markdown, no code blocks,
# using only the page text.



In [ ]:
# Step 3 - Pick one company: build the prompt (name + scraped text) and print it.



In [ ]:
# Step 4 - Write stream_pamphlet(company_name, url): call completion(...) with
# stream=True and yield the text as it grows.



In [ ]:
# Step 5 - Put it in a gr.Interface (name + URL in, markdown out) and launch it.



In [ ]:
# Step 6 (stretch) - Add a tone dropdown: professional / playful / recruiter-facing.



---

**When it misbehaves:**

| What you see | What it means |
|---|---|
| `Error fetching website: ...` | the site blocked the request, or the URL is missing `https://` — try another company |
| `AuthenticationError` | the environment variable never got set — re-run your Step 1 cell and paste the key again |
| The pamphlet appears all at once | your function `return`s instead of `yield`ing — a streaming UI needs a generator |
| Facts that are nowhere on the page | the model filled the gaps itself. 2,000 characters is a small window — say *"use only the page text"* in the system message |
| `BadRequestError: LLM Provider NOT provided` | the model string needs its prefix: `openai/gpt-4o-mini`, not `gpt-4o-mini` |

---

### ✅ What you practised

| Idea | The one-liner |
|---|---|
| **`requests` + BeautifulSoup** | scraping is plain Python — the model only sees text you hand it |
| **`getpass` + env var** | the key is typed at runtime, never written into the file |
| **LiteLLM `completion()`** | one function for every provider — the model string is the only thing that changes |
| **`stream=True` + `yield`** | every yield repaints the output box, so the pamphlet types out live |
| **`gr.Interface`** | two textboxes and a markdown box — that is the entire front end |

**Finished early?**
1. Raise `max_chars` to 5,000 — does the pamphlet get better, or just longer?
2. Swap the model to `gemini/gemini-2.0-flash` (set `GEMINI_API_KEY` instead) — nothing else changes.
3. Point it at your own college and tune the system message until the output reads like
   something you would actually hand to someone.